<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_5_rag/week5_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 (starter): RAG Pipeline with Retrieval Evaluation

Retrieval runs fully local and free; only generation needs a key (set `GEMINI_API_KEY`, else it is skipped with a notice). Cells marked **TODO (you)** are yours. Dependencies: `sentence-transformers`, `faiss-cpu`. For generation: `pip install openai`.

In [1]:
%pip install -q sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.9 MB/s eta 0:00:00


In [2]:
import os, re, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# TODO (you): done -- 8 short docs for a fictional server product (Orbit), replacing the
# starter's single placeholder DOC. Facts are written plainly so relevance can be judged
# by whether a fact string appears in a chunk (see Part 2).
DOCS = {
    'networking': '''## Networking
The Orbit server listens on TCP port 7700 by default. Gossip runs on 7711.
Clients should connect to port 7700; port 7711 is for inter-node gossip only
and should not be exposed publicly.''',

    'storage': '''## Storage
Orbit persists data with an LSM-tree storage engine.
Writes go to an in-memory memtable first and are flushed to disk as
immutable SSTable segments once the memtable fills.''',

    'replication': '''## Replication
Each key is replicated with a default replication factor of 3.
Default consistency is quorum, meaning a write or read must be
acknowledged by at least two of the three replicas to succeed.''',

    'limits': '''## Limits
The maximum value size is 16 MB. A batch may contain at most 1000 operations.
Keys are limited to 1 KB. Requests exceeding any of these limits are
rejected with a client-side validation error before reaching storage.''',

    'security': '''## Security
Clients authenticate with API tokens. Traffic is encrypted with TLS.
Tokens are scoped per-namespace and can be revoked instantly through
the admin API without restarting the server.''',

    'backup': '''## Backup
Orbit takes incremental snapshots every 6 hours by default, written to
the configured backup directory. A full snapshot can be triggered manually
with orbitctl snapshot --full. Snapshots older than 7 days are pruned
automatically unless retention is overridden in orbit.yaml.''',

    'monitoring': '''## Monitoring
Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics.
Health checks are available at /healthz on the same port. The metrics
port is separate from the client port and is not encrypted by default.''',

    'deployment': '''## Deployment
Orbit is configured through a single YAML file, orbit.yaml, loaded at
startup. Environment variables prefixed with ORBIT_ override values from
the file. The default data directory is /var/lib/orbit, and the default
config search path is /etc/orbit/orbit.yaml.''',
}
print('doc count:', len(DOCS), '| total chars:', sum(len(t) for t in DOCS.values()))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

doc count: 8 | total chars: 1789


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

def build_config(chunk_fn, *args):
    """Run a chunker over every document, returning a flat chunk list plus, for each
    chunk, which source doc it came from -- kept for Part 2's relevance labels and for
    tracing exactly where a fact came from in Part 4."""
    chunks, sources = [], []
    for name, text in DOCS.items():
        pieces = chunk_fn(text, *args) if args else chunk_fn(text)
        for c in pieces:
            chunks.append(c); sources.append(name)
    return chunks, sources

configs = {
    'small (120/20)': build_config(chunk_fixed, 120, 20),
    'large (320/40)': build_config(chunk_fixed, 320, 40),
    'by-paragraph':   build_config(chunk_paragraph),
}
for n, (c, s) in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    e = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
    idx = faiss.IndexFlatIP(e.shape[1]); idx.add(e); return idx
def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    # FAISS uses -1 for missing results when fewer than k chunks exist.
    ids = idx.search(qe, k)[1][0]
    return [chunks[i] for i in ids if i >= 0], [int(i) for i in ids if i >= 0]

indices = {n: build(c) for n, (c, s) in configs.items()}

small (120/20)   -> 22 chunks
large (320/40)   -> 9 chunks
by-paragraph     -> 8 chunks


## Part 2: Evaluate retrieval (compare chunking configs)
A query is answered only if the retrieved context contains the answer fact. This is the model-free half of answer quality.

In [4]:
# TODO (you): done. >= 10 queries, each with a fact decided in advance and the doc that
# fact belongs to (used only for reference in the write-up, not for scoring -- scoring uses
# the fact string, per the ground-truth rule described below).
QA = [
    ('what port does orbit listen on for clients', '7700', 'networking'),
    ('what port is used for gossip between nodes', '7711', 'networking'),
    ('what storage engine does orbit use', 'LSM-tree', 'storage'),
    ('what happens when the memtable fills up', 'flushed to disk', 'storage'),
    ('what is the default replication factor', 'replication factor of 3', 'replication'),
    ('what consistency level is used by default', 'quorum', 'replication'),
    ('what is the maximum value size', '16 MB', 'limits'),
    ('how many operations can a batch contain', '1000 operations', 'limits'),
    ('how do clients authenticate', 'API tokens', 'security'),
    ('how often are snapshots taken', 'every 6 hours', 'backup'),
    ('what port serves metrics', '7720', 'monitoring'),
    ('where can i check server health', '/healthz', 'monitoring'),
    ('how is orbit configured', 'orbit.yaml', 'deployment'),
    ('what environment variable prefix overrides config', 'ORBIT_', 'deployment'),
]
print(f'{len(QA)} queries defined (>= 10 required)')

def chunk_relevance(chunks, fact):
    """Ground truth, decided in advance: a chunk counts as relevant to a query iff it
    contains that query's pre-written answer fact. The fact strings above were written
    down before any chunking config was picked, so the *criterion* is fixed in advance,
    even though the resulting index set is necessarily computed per config (chunk
    boundaries differ between configs)."""
    return {i for i, c in enumerate(chunks) if fact.lower() in c.lower()}

def precision_recall_at_k(config_name, k=3):
    chunks, sources = configs[config_name]
    idx = indices[config_name]
    p_sum, r_sum, n = 0.0, 0.0, 0
    for q, fact, doc in QA:
        _, retrieved_ids = retrieve(idx, chunks, q, k=k)
        retrieved = set(retrieved_ids)
        relevant = chunk_relevance(chunks, fact)
        if not relevant:
            print(f'WARNING: fact {fact!r} matched 0 chunks in {config_name} -- skipped')
            continue
        tp = len(retrieved & relevant)
        p_sum += tp / len(retrieved) if retrieved else 0.0
        r_sum += tp / len(relevant)
        n += 1
    return p_sum / n, r_sum / n

for name in configs:
    p, r = precision_recall_at_k(name, k=3)
    print(f'{name:16s} precision@3={p:.2f}  recall@3={r:.2f}')

14 queries defined (>= 10 required)
small (120/20)   precision@3=0.40  recall@3=0.94
large (320/40)   precision@3=0.36  recall@3=1.00
by-paragraph     precision@3=0.36  recall@3=1.00


**Results.** *(Fill in once the cell above has run with the real embedding model: which
config had the best precision, which had the best recall, and why? The corpus is designed so
`small (120/20)` fragments several sentences mid-clause -- see the confirmed splits in Part 4
-- while `large (320/40)` and `by-paragraph` keep every sentence in this corpus whole in a
single chunk. Report what the real embeddings actually show, not what's expected -- semantic
retrieval doesn't always penalize a split chunk the way exact substring matching would.)*

## Part 3: Generate (needs a key)
Build a grounded prompt from the retrieved chunks and answer with Gemini.

In [5]:
def _get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if key:
        return key
    try:
        import getpass
        return getpass.getpass('ANTHROPIC_API_KEY (leave blank to skip generation): ').strip()
    except Exception:
        return ''

_API_KEY = _get_api_key()
if _API_KEY:
    import anthropic
    _client = anthropic.Anthropic(api_key=_API_KEY)
GEN_MODEL = 'claude-haiku-4-5-20251001'

def claude_chat(messages, model=GEN_MODEL, **kw):
    if not _API_KEY:
        return None
    return _client.messages.create(model=model, max_tokens=512, messages=messages, **kw).content[0].text

def answer(q, config='by-paragraph', k=3):
    chunks, sources = configs[config]
    ctx, ids = retrieve(indices[config], chunks, q, k=k)
    prompt = 'Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) + f'\nQuestion: {q}\nAnswer:'
    out = claude_chat([{'role':'user','content':prompt}])
    return (out if out is not None else '[API-BLOCKED] set ANTHROPIC_API_KEY to generate; grounded prompt was built'), ctx

resp, ctx = answer('what port does orbit use')
print(resp)

Based on the context, Orbit uses the following ports:

- **TCP port 7700** - Default port for client connections
- **Port 7711** - Used for inter-node gossip communication (should not be exposed publicly)
- **Port 7720** - Used for Prometheus-compatible metrics and health checks


## Part 4 and 5: failure, submit
**TODO (you):** show a query where retrieval surfaces the wrong chunk or a chunk config splits a fact, and explain the mitigation. Then open a pull request with a result summary and review a classmate's Week 5 PR (the term's formal peer review). Use the rubric attached to the Canvas assignment for grading criteria and point values.